In [100]:
from pathlib import Path
import sys
sys.path.append(str(Path.cwd().parent))

from src.data_loader import load_data
import src.features
import src.model
import src.preprocessing

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import importlib
importlib.reload(src.preprocessing)
importlib.reload(src.model)
importlib.reload(src.features);

Load data and preprocessing steps(split data, remove invalid values, handle outlier values, impute missing values):

In [101]:
PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "cs-training.csv"

TARGET = "SeriousDlqin2yrs"

BASELINE_FEATURES = [
    "RevolvingUtilizationOfUnsecuredLines",
    "age",
    "NumberOfTime30-59DaysPastDueNotWorse",
    "DebtRatio",
    "MonthlyIncome",
    "NumberOfOpenCreditLinesAndLoans",
    "NumberOfTimes90DaysLate",
    "NumberRealEstateLoansOrLines",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfDependents"
]  

df = load_data(DATA_PATH)
train_df, test_df = src.preprocessing.split_data(df, test_size=0.2, random_state=67)

train_df = src.preprocessing.remove_invalid_ages(train_df)
test_df = src.preprocessing.remove_invalid_ages(test_df)

train_df = src.preprocessing.handle_sentinel_values(train_df)
test_df = src.preprocessing.handle_sentinel_values(test_df)

train_df, test_df = src.preprocessing.handle_debt_ratio_outlier(train_df,test_df)

train_df, test_df = src.preprocessing.handle_revolving_utilization_outlier(train_df, test_df)

train_df, test_df = src.preprocessing.impute_missing(train_df=train_df, 
                                       test_df=test_df, 
                                       col="MonthlyIncome")
train_df, test_df = src.preprocessing.impute_missing(train_df=train_df, 
                                       test_df=test_df, 
                                       col="NumberOfDependents")

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")


Train shape: (119792, 13)
Test shape: (29938, 13)


In [102]:
X_train, y_train = src.model.prepare_features(train_df, TARGET, BASELINE_FEATURES)
X_test, y_test = src.model.prepare_features(test_df, TARGET, BASELINE_FEATURES)

baseline_model = src.model.fit_logistic_model(train_df,TARGET,BASELINE_FEATURES)

baseline_train_prob = src.model.predict_probabilities(baseline_model, train_df,BASELINE_FEATURES)
baseline_test_prob = src.model.predict_probabilities(baseline_model, test_df,BASELINE_FEATURES)
# print(train_prob)
# print(test_prob)

No evidence of overfitting; both metrics are close enough together

In [105]:
train_metrics = src.model.evaluate_model(y_train, baseline_train_prob)
test_metrics = src.model.evaluate_model(y_test, baseline_test_prob)
print(f"Baseline Train metrics: {train_metrics}")
print(f"Baseline Test metrics: {test_metrics}")

Baseline Train metrics: {'ROC-AUC': 0.8493214359552966, 'KS': np.float64(0.5399748238263369)}
Baseline Test metrics: {'ROC-AUC': 0.8519634610646127, 'KS': np.float64(0.5457656181241559)}
